# AncestryClassifier — Colab Training

Run preprocessing locally with Snakemake first:
```bash
snakemake --cores 4 prepare_training_data simulate_admixed
```
Upload `data/dataset.h5` and `data/admixed_test.h5` to Google Drive, then set `DRIVE_DIR` below.

**After any runtime restart: re-run Cell 1 (config) before running any other cell.**

In [8]:
# ── Cell 1: config — re-run this first after every runtime restart ─────────
from google.colab import drive
drive.mount('/content/drive')

DRIVE_DIR     = '/content/drive/MyDrive/gene461'          # <-- change if needed
REPO          = '/content/gene_461_final_project'
WINDOW_SIZE   = 1000

DATA_H5       = f'{DRIVE_DIR}/dataset.h5'
ADMIXED_H5    = f'{DRIVE_DIR}/admixed_test.h5'
CKPT_OUT      = f'{DRIVE_DIR}/checkpoints/best_model.pt'
CONFUSION_OUT = f'{DRIVE_DIR}/confusion_matrix.png'
KARYOGRAM_OUT = f'{DRIVE_DIR}/lai_karyogram.png'

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [9]:
# ── Cell 2: one-time setup (clone repo + install deps) ────────────────────
import subprocess, os

if not os.path.exists(REPO):
    subprocess.run(
        ['git', 'clone', 'https://github.com/aszatrowski/gene_461_final_project', REPO],
        check=True
    )
else:
    subprocess.run(['git', '-C', REPO, 'pull'], check=True)

%pip install -q torch h5py numpy pandas scikit-learn matplotlib wandb

In [10]:
# ── Cell 3: verify GPU ─────────────────────────────────────────────────────
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

CUDA available: True
GPU: Tesla T4


In [ ]:
# ── Cell 4: baseline train (needs Cell 1) ─────────────────────────────────
import os
os.makedirs(f'{DRIVE_DIR}/checkpoints', exist_ok=True)

!python {REPO}/scripts/train.py \
    --data        {DATA_H5}    \
    --output      {CKPT_OUT}   \
    --window-size {WINDOW_SIZE} \
    --epochs      25           \
    --batch-size  512          \
    --num-workers 2

Device: cuda
Loading training data ...


^C


In [14]:
# ── Cell 5: W&B login (one-time per session) ───────────────────────────────
import wandb
wandb.login()

True

In [15]:
# ── Cell 6: sweep config ───────────────────────────────────────────────────
sweep_config = {
    'method': 'bayes',
    'metric': {'name': 'val_acc', 'goal': 'maximize'},
    'parameters': {
        'window_size': {'values': [500, 1000, 2000]},
        'lr':          {'distribution': 'log_uniform_values', 'min': 1e-4, 'max': 1e-2},
        'conv_arch':   {'values': [
            '32,64_7,5',          # baseline (already ran)
            '64,128_7,5',         # wider
            '32,64,128_7,5,3',    # deeper
            '64,128,256_7,5,3',   # wider + deeper
            '32_7',               # 1-block shallow
            '128,256_11,7',       # large receptive field
            '32,64_11,7',         # baseline channels, bigger kernels
            '64,128,256_11,7,5',  # widest
        ]},
        'dropout': {'distribution': 'uniform', 'min': 0.1, 'max': 0.5},
    },
}

sweep_id = wandb.sweep(sweep_config, project='ancestry_cnn')
print('Sweep ID:', sweep_id)

Create sweep with ID: 6ps21dz6
Sweep URL: https://wandb.ai/aszatrowski-university-of-chicago/ancestry_cnn/sweeps/6ps21dz6
Sweep ID: 6ps21dz6


In [ ]:
# ── Cell 7: run sweep agent (needs Cells 1, 3, 5, 6) ─────────────────────
import sys
sys.path.insert(0, f'{REPO}/scripts')
# Evict both modules so Python re-reads the updated files from disk
sys.modules.pop('train', None)
sys.modules.pop('model', None)
from train import run_training, parse_conv_arch

SWEEP_EPOCHS = 10   # short runs during search; best config retrained with 25 epochs below
SWEEP_COUNT  = 20   # total trials (~3-4 h on T4)

def sweep_fn():
    with wandb.init() as run:
        w = dict(run.config)
        channels, kernels = parse_conv_arch(w['conv_arch'])
        cfg = {
            'window_size':   w['window_size'],
            'lr':            w['lr'],
            'conv_channels': channels,
            'kernel_sizes':  kernels,
            'dropout':       w['dropout'],
            'epochs':        SWEEP_EPOCHS,
            'batch_size':    512,
            'num_workers':   2,
            'use_wandb':     True,
        }
        run_training(cfg, DATA_H5, None, device)  # output=None skips checkpoint saving

wandb.agent(sweep_id, sweep_fn, count=SWEEP_COUNT)

In [ ]:
# ── Cell 8: retrain best config for full 25 epochs ────────────────────────
api   = wandb.Api()
sweep = api.sweep(f'aszatrowski-university-of-chicago/ancestry_cnn/{sweep_id}')
best  = max(sweep.runs, key=lambda r: r.summary.get('val_acc', 0))
print('Best run:', best.name)
print('Config:  ', dict(best.config))
print(f'val_acc:  {best.summary["val_acc"]:.4f}')

bc = dict(best.config)
channels, kernels = parse_conv_arch(bc['conv_arch'])
best_cfg = {
    'window_size':   bc['window_size'],
    'lr':            bc['lr'],
    'conv_channels': channels,
    'kernel_sizes':  kernels,
    'dropout':       bc['dropout'],
    'epochs':        25,
    'batch_size':    512,
    'num_workers':   2,
    'use_wandb':     True,
}
run_training(best_cfg, DATA_H5, CKPT_OUT, device)

In [ ]:
# ── Cell 9: evaluate (needs Cell 1 only — safe after a restart) ───────────
!python {REPO}/scripts/evaluate.py \
    --data        {DATA_H5}       \
    --admixed     {ADMIXED_H5}    \
    --checkpoint  {CKPT_OUT}      \
    --confusion   {CONFUSION_OUT} \
    --karyogram   {KARYOGRAM_OUT} \
    --window-size {WINDOW_SIZE}

In [ ]:
# ── Cell 10: display results (needs Cell 1 only) ──────────────────────────
from IPython.display import Image, display
display(Image(CONFUSION_OUT))
display(Image(KARYOGRAM_OUT))